# 09 · The prism

Notebooks 01–08 used the **G150 grism**. WFI carries a second dispersing element, the **P127 prism**: a single spectral order covering **0.75–1.85 µm**, with a strongly wavelength-dependent — and much lower — spectral resolution, trading resolution for blue coverage. Since v0.14.0 `roman_disperser` treats the element as *data, not code*: a frozen `DispersingElement` record carries everything element-specific (orders, band edges, STPSF filter per order, sensitivity directory), and every entry point takes `element=`.

This notebook swaps `PRISM` in for `GRISM` and shows the workflow of the whole series carries over unchanged — then uses the optical model itself to *measure* what differs: the trace, the dispersion, and the resolving power. Nothing here is new machinery; it is the machinery of notebooks 02, 04 and 07 pointed at a different element.

> **Needs** the prism reference data hydrated (`pixi run hydrate` fetches both elements; notebook 00 checks them).

## 0 · Setup — the two elements side by side

The `ELEMENTS` registry maps names to records; `get_element()` resolves a name (or `None` → grism) and raises on a typo instead of silently defaulting. Printing both records shows the whole element-level difference between the two instruments.

In [ ]:
import os
from pathlib import Path
os.environ.setdefault("JAX_COMPILATION_CACHE_DIR", str(Path.home() / ".cache" / "roman_grs_jax"))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import AsinhNorm
import jax
import jax.numpy as jnp

from roman_disperser import paths, psf_model, star_disperser, elements
from roman_disperser.elements import GRISM, PRISM
from roman_disperser.optical_model import RomanOpticalModel
import roman_disperser.optical_model_jax as omj
import tutorial_helpers as th

SCA = 5
DETECTOR = f"WFI{SCA:02d}"
print("registry:", list(elements.ELEMENTS), "| JAX backend:", jax.default_backend())
for el in (GRISM, PRISM):
    print(f"  {el.name:6s}: orders {el.orders}, band {el.lam_min}–{el.lam_max} µm, "
          f"STPSF filters {el.stpsf_filters}, sensitivities '{el.sensitivities_subdir}'")

## 1 · Per-element reference data, checked not assumed

Each element has its own optical-model delivery (resolved from the hydrated data directory's `data-versions.lock`) and its own sensitivity directory; the PSF caches share one directory because the filenames carry the STPSF filter name. The pairing between an element and an optical model is *validated*, not assumed — `elements.validate_against_model` compares the element's name and nominal band against what the delivery declares, so loading the wrong YAML for an element fails loudly instead of producing silently wrong traces.

In [ ]:
model = {el.name: RomanOpticalModel(config_file=str(paths.optical_model_path(element=el)))
         for el in (GRISM, PRISM)}
opt = {el.name: omj.make_sca_payload(model[el.name], sca=SCA, order="1")
       for el in (GRISM, PRISM)}
for el in (GRISM, PRISM):
    elements.validate_against_model(el, model[el.name])      # silent when consistent
print("both optical models load and validate against their elements")

# the pairing is checked: mixing them up raises rather than mis-tracing
try:
    elements.validate_against_model(PRISM, model["grism"])
except ValueError as e:
    print("\nmixed pairing raises:\n  ", e)

## 2 · The trace — same star, two elements

The coordinate machinery is identical (notebook 07): `sca_to_fpa → trace_beam → mpa_to_sca`, per element via its own payload. We trace the *same* undispersed source position through both and compare where each wavelength lands.

In [ ]:
X0, Y0 = 2000.0, 2000.0
wl = {el.name: jnp.linspace(el.lam_min, el.lam_max, 300) for el in (GRISM, PRISM)}
trace = {}
for el in (GRISM, PRISM):
    p = opt[el.name]
    xf, yf = omj.sca_to_fpa(p, jnp.array([X0]), jnp.array([Y0]))
    xm, ym = omj.trace_beam(p, jnp.broadcast_to(xf, wl[el.name].shape),
                            jnp.broadcast_to(yf, wl[el.name].shape), wl[el.name])
    tx, ty = omj.mpa_to_sca(p, xm, ym)
    trace[el.name] = (np.asarray(tx), np.asarray(ty))

fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 3.8))
for name, color in (("grism", "C1"), ("prism", "C0")):
    a0.plot(np.asarray(wl[name]), trace[name][1], color=color, label=name)
    a1.plot(trace[name][0], trace[name][1], color=color, label=name)
a0.set(xlabel="wavelength [µm]", ylabel="trace y [pix]", title="where each wavelength lands")
a1.plot(X0, Y0, "k+", ms=10, label="source")
a1.set(xlabel="trace x [pix]", ylabel="trace y [pix]", title="order-1 traces on the detector")
a0.legend(); a1.legend()
fig.tight_layout()
for name in ("grism", "prism"):
    ty = trace[name][1]
    print(f"{name:6s}: order-1 trace spans {abs(ty.max() - ty.min()):6.0f} px")

## 3 · Dispersion and resolving power, by autodiff

As in notebooks 04 and 07, the local dispersion is `jax.grad` of the trace. From it we form a **per-pixel resolving power**

$$R \;=\; \frac{\lambda}{\Delta\lambda_{\rm pix}} \;=\; \lambda \left|\frac{dy}{d\lambda}\right|,$$

where $\Delta\lambda_{\rm pix} = |dy/d\lambda|^{-1}$ is the wavelength interval spanned by one pixel along the trace. This is a *sampling* definition — a real resolution element is set by the PSF, not the pixel grid. For the grism, the mission-standard resolving power is $R \approx 461\,(\lambda/1.45\,\mu\mathrm{m})$ *per resolution element*, which corresponds to a ~3-pixel element (≈1.5× the PSF FWHM): dividing the per-pixel $R$ measured below by that width recovers the mission value to a few percent. The comparison is the point: the grism's dispersion is constant to about a percent across its band (a grating), while the prism's varies by a factor of ~4–5 across its band (glass dispersion is strongly chromatic) — the two differ in normalization *and* shape.

In [ ]:
def make_trace_y(payload):
    def trace_y(wl_scalar):
        w = jnp.atleast_1d(wl_scalar)
        xf, yf = omj.sca_to_fpa(payload, jnp.array([X0]), jnp.array([Y0]))
        xm, ym = omj.trace_beam(payload, jnp.broadcast_to(xf, w.shape),
                                jnp.broadcast_to(yf, w.shape), w)
        _, ty = omj.mpa_to_sca(payload, xm, ym)
        return ty[0]
    return trace_y

fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 3.8))
for name, color in (("grism", "C1"), ("prism", "C0")):
    w = np.asarray(wl[name])
    dydl = np.abs(np.asarray(jax.jit(jax.vmap(jax.grad(make_trace_y(opt[name]))))(wl[name])))
    a0.semilogy(w, dydl, color=color, label=name)
    a1.semilogy(w, w * dydl, color=color, label=name)
    i = np.argmin(np.abs(w - 1.45))
    print(f"{name:6s}: |dy/dλ| = {dydl[i]:6.0f} pix/µm at 1.45 µm "
          f"→ per-pixel R ≈ {1.45 * dydl[i]:5.0f}")
a0.set(xlabel="wavelength [µm]", ylabel="|dy/dλ| [pix/µm]", title="dispersion")
a1.set(xlabel="wavelength [µm]", ylabel="R = λ·|dy/dλ|", title="per-pixel resolving power")
a0.legend(); a1.legend()
fig.tight_layout()

## 4 · Disperse the same star with both elements

The pipeline of notebook 02, verbatim, once per element: a count-rate spectrum on the element's band (`template_to_counts(..., element=...)` picks the band *and* the right sensitivity directory), a PSF payload derived from the element (`element=` sets the STPSF filter and the PSF wavelength grid together — pass it rather than juggling `stpsf_filter=`/`wavelengths=` by hand), and the same compiled star disperser.

In [ ]:
imgs, counts = {}, {}
for el in (GRISM, PRISM):
    wl_um, _, _ = th.wavelength_grid(el)
    _, c = th.template_to_counts("g0v", 18.0, sca=SCA, order="1", wl_um=wl_um, element=el)
    counts[el.name] = (wl_um, c)
    psf = psf_model.get_or_make_psf_payload(detector=DETECTOR, order="1", element=el,
                                            cache_dir=str(paths.psf_cache_dir()), verbose=False)
    disp = star_disperser.make_star_disperser(psf, opt[el.name])
    img = disp(X0, Y0, jnp.asarray(wl_um), jnp.asarray(c),
               jnp.zeros((4088, 4088), jnp.float32))
    img.block_until_ready()
    imgs[el.name] = np.asarray(img)
    print(f"{el.name:6s}: input {c.sum():7.1f} e-/s over {wl_um.size} samples; "
          f"deposited {imgs[el.name].sum():7.1f} e-/s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.5, 6.5))
for ax, name in zip(axes, ("grism", "prism")):
    img = imgs[name]
    ys, xs = np.nonzero(img)
    pad = 12
    cut = img[ys.min()-pad:ys.max()+pad, xs.min()-pad:xs.max()+pad]
    ax.imshow(cut, origin="lower", cmap="inferno",
              norm=AsinhNorm(linear_width=cut.max()*0.01, vmin=0, vmax=cut.max()))
    ax.set(title=f"{name} · order 1", xticks=[], yticks=[])
fig.suptitle("the same G0V star (AB = 18), dispersed by each element")
fig.tight_layout()

The prism packs its whole band into a much shorter trace — the same source's light lands on far fewer pixels, which is the per-pixel-depth side of the resolution trade.

## 5 · Sensitivity and counts

The per-element sensitivity curves close the loop: same detector, same star, different element. The count-rate spectra differ both through the sensitivity (throughput into order 1) and through the band each element sees.

In [ ]:
fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 3.6))
for name, color in (("grism", "C1"), ("prism", "C0")):
    wl_um, c = counts[name]
    sens = th.load_sensitivity(SCA, "1", wl_um * 1e4, element=name)
    a0.plot(wl_um, sens, color=color, label=name)
    a1.plot(wl_um, c, color=color, label=name)
a0.set(xlabel="wavelength [µm]", ylabel="sensitivity [counts/s per FLAM·Å]",
       title=f"order-1 sensitivity, SCA{SCA}")
a1.set(xlabel="wavelength [µm]", ylabel="count rate [e⁻/s per 2 Å bin]",
       title="G0V, AB = 18 → counts")
a0.legend(); a1.legend()
fig.tight_layout()

## Recap

- The element swap **is** the whole migration: the payload → PSF → disperser pipeline of notebooks 02–08 runs unchanged with `element=PRISM`; the record supplies the single order, the 0.75–1.85 µm band, the `PRISM` STPSF filter, and `sensitivities_prism/`.
- The optical model *measures* the differences: the prism's short, strongly chromatic trace versus the grism's long, nearly uniform one — quantified by the same autodiff dispersion used throughout the series.
- The element/optical-model pairing is **validated** (`validate_against_model`), and a wrong `element=`/data combination raises rather than mis-placing traces.
- At production scale the same switch is one flag: `scripts/build_dispersed_image.py --element prism`, which writes `prism_*` outputs stamped with an `OPTELEM` header card.

This closes the loop on the series: one differentiable optical-model core (07), one dispersion pipeline (02–06, 08), two instruments (09).